# Voice Assistant with MiniMax TTS and Claude

This notebook demonstrates how to build a voice assistant by combining:

- **[Claude](https://www.anthropic.com/claude)** — intelligent text generation
- **[MiniMax TTS](https://platform.minimax.io/docs/api-reference/speech-t2a-http)** — high-quality text-to-speech synthesis

MiniMax provides a streaming TTS API that delivers audio as Server-Sent Events (SSE) with hex-encoded audio chunks, enabling low-latency speech synthesis. We'll cover:

1. Basic (non-streaming) TTS with MiniMax
2. Streaming TTS with SSE parsing
3. A complete voice assistant combining Claude responses with MiniMax TTS

## Prerequisites

- An [Anthropic API key](https://console.anthropic.com/settings/keys)
- A [MiniMax API key](https://platform.minimax.io/)

In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install -r requirements.txt

## Imports

In [ ]:
import json
import os

import anthropic
import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

## API Keys

Set up your API keys for MiniMax and Anthropic.

**Setup Instructions:**

1. Create a `.env` file in this directory
2. Add your API keys:
   ```
   MINIMAX_API_KEY=your_minimax_api_key_here
   ANTHROPIC_API_KEY=sk-ant-api03-...
   ```
3. Get your MiniMax API key: https://platform.minimax.io/
4. Get your Anthropic API key: https://console.anthropic.com/settings/keys

In [ ]:
load_dotenv()

MINIMAX_API_KEY = os.getenv("MINIMAX_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

In [ ]:
assert MINIMAX_API_KEY is not None, (
    "ERROR: MINIMAX_API_KEY not found. Please create a .env file and add your MiniMax API key."
)
assert ANTHROPIC_API_KEY is not None, (
    "ERROR: ANTHROPIC_API_KEY not found. Please create a .env file and add your Anthropic API key."
)

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MINIMAX_BASE_URL = "https://api.minimax.io"

print("Clients initialized successfully.")

## MiniMax TTS Voices

MiniMax offers several English voices with different personalities:

In [ ]:
MINIMAX_VOICES = [
    "English_Graceful_Lady",
    "English_Insightful_Speaker",
    "English_radiant_girl",
    "English_Persuasive_Man",
    "English_Lucky_Robot",
    "English_expressive_narrator",
]

print("Available MiniMax TTS voices:")
for voice in MINIMAX_VOICES:
    print(f"  - {voice}")

## Basic Text-to-Speech

The MiniMax TTS API accepts text and returns audio in MP3 format. The audio data is hex-encoded in the response. We'll start with the non-streaming approach for simplicity.

In [ ]:
def text_to_speech(
    text: str,
    voice: str = "English_Graceful_Lady",
    model: str = "speech-2.8-hd",
) -> bytes:
    """Convert text to speech using MiniMax TTS API (non-streaming).

    Args:
        text: The text to synthesize.
        voice: Voice ID to use for synthesis.
        model: TTS model to use. Options: 'speech-2.8-hd' (default) or 'speech-2.8-turbo'.

    Returns:
        MP3 audio data as bytes.
    """
    url = f"{MINIMAX_BASE_URL}/v1/t2a_v2"
    headers = {
        "Authorization": f"Bearer {MINIMAX_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "text": text,
        "stream": False,
        "voice_setting": {
            "voice_id": voice,
            "speed": 1,
            "vol": 1,
            "pitch": 0,
        },
        "audio_setting": {
            "sample_rate": 32000,
            "bitrate": 128000,
            "format": "mp3",
            "channel": 1,
        },
    }

    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()

    result = response.json()
    status_code = result["base_resp"]["status_code"]
    assert status_code == 0, f"TTS error {status_code}: {result['base_resp']['status_msg']}"

    # Audio data is hex-encoded (not base64)
    return bytes.fromhex(result["data"]["audio"])

In [ ]:
# Generate a test audio clip
test_text = "Hello! I'm powered by MiniMax TTS and Claude. Let's build something amazing together."
print(f"Synthesizing: {test_text!r}")

audio_bytes = text_to_speech(test_text)
print(f"Generated {len(audio_bytes):,} bytes of MP3 audio")

display(Audio(audio_bytes, rate=32000, autoplay=False))

## Streaming Text-to-Speech

For longer texts, streaming reduces time-to-first-audio by delivering audio chunks as they're generated. MiniMax uses Server-Sent Events (SSE) format where each `data:` line contains a JSON object with a hex-encoded audio chunk.

**Important**: MiniMax TTS audio is hex-encoded, not base64. Use `bytes.fromhex()` to decode.

In [ ]:
def text_to_speech_streaming(
    text: str,
    voice: str = "English_Graceful_Lady",
    model: str = "speech-2.8-hd",
) -> bytes:
    """Convert text to speech using MiniMax TTS API with SSE streaming.

    MiniMax streams audio as Server-Sent Events. Each 'data:' line contains
    a JSON object with a hex-encoded audio chunk in data.audio.

    Args:
        text: The text to synthesize.
        voice: Voice ID to use for synthesis.
        model: TTS model to use.

    Returns:
        Complete MP3 audio data as bytes.
    """
    url = f"{MINIMAX_BASE_URL}/v1/t2a_v2"
    headers = {
        "Authorization": f"Bearer {MINIMAX_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "text": text,
        "stream": True,
        "voice_setting": {
            "voice_id": voice,
            "speed": 1,
            "vol": 1,
            "pitch": 0,
        },
        "audio_setting": {
            "sample_rate": 32000,
            "bitrate": 128000,
            "format": "mp3",
            "channel": 1,
        },
    }

    response = requests.post(url, headers=headers, json=payload, stream=True, timeout=120)
    response.raise_for_status()

    audio_chunks: list[bytes] = []
    chunk_count = 0

    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue

        data_str = line[5:].strip()  # Remove 'data:' prefix
        if not data_str or data_str == "[DONE]":
            continue

        try:
            event = json.loads(data_str)
            audio_hex = event.get("data", {}).get("audio", "")
            if audio_hex:
                audio_chunks.append(bytes.fromhex(audio_hex))  # hex, not base64
                chunk_count += 1
        except (json.JSONDecodeError, ValueError):
            pass

    print(f"Received {chunk_count} audio chunks")
    return b"".join(audio_chunks)

In [ ]:
# Compare streaming vs non-streaming
long_text = (
    "The cosmos is vast and filled with wonders beyond imagination. "
    "From the swirling nebulae where stars are born, to the mysterious black holes "
    "that warp the fabric of spacetime, the universe never ceases to amaze us."
)

print("Streaming TTS...")
audio_streaming = text_to_speech_streaming(long_text)
print(f"Total audio size: {len(audio_streaming):,} bytes")

display(Audio(audio_streaming, rate=32000, autoplay=False))

## Voice Assistant: Claude + MiniMax TTS

Now let's combine Claude's text generation with MiniMax's TTS to build a voice assistant. Claude generates the response, and MiniMax speaks it aloud.

In [ ]:
def voice_assistant(
    user_message: str,
    voice: str = "English_Graceful_Lady",
    system_prompt: str = "You are a helpful and friendly voice assistant. Keep your responses concise and conversational, ideally 2-3 sentences.",
) -> tuple[str, bytes]:
    """Generate a Claude response and convert it to speech with MiniMax TTS.

    Args:
        user_message: The user's input message.
        voice: MiniMax voice ID to use.
        system_prompt: System prompt for Claude.

    Returns:
        Tuple of (response_text, audio_bytes).
    """
    # Step 1: Get Claude's response
    message = claude_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    response_text = message.content[0].text

    # Step 2: Convert response to speech
    audio_bytes = text_to_speech_streaming(response_text, voice=voice)

    return response_text, audio_bytes

In [ ]:
# Example 1: General knowledge question
user_input = "What are three fascinating facts about the deep ocean?"
print(f"User: {user_input}\n")

response_text, audio_bytes = voice_assistant(user_input)
print(f"Claude: {response_text}")

display(Audio(audio_bytes, rate=32000, autoplay=False))

In [ ]:
# Example 2: Try a different voice and topic
user_input = "Tell me a short, uplifting story about a robot learning to paint."
print(f"User: {user_input}\n")

response_text, audio_bytes = voice_assistant(user_input, voice="English_Lucky_Robot")
print(f"Claude: {response_text}")

display(Audio(audio_bytes, rate=32000, autoplay=False))

## Streaming Voice Assistant

For a more responsive experience, we can stream Claude's response text as it's generated. This lets us display text progressively while also preparing for TTS synthesis.

In [ ]:
def voice_assistant_streaming(
    user_message: str,
    voice: str = "English_Graceful_Lady",
    system_prompt: str = "You are a helpful and friendly voice assistant. Keep your responses concise and conversational, ideally 2-3 sentences.",
) -> tuple[str, bytes]:
    """Stream Claude's response and convert the full text to speech with MiniMax TTS.

    Args:
        user_message: The user's input message.
        voice: MiniMax voice ID to use.
        system_prompt: System prompt for Claude.

    Returns:
        Tuple of (response_text, audio_bytes).
    """
    # Step 1: Stream Claude's response
    print("Claude: ", end="", flush=True)
    full_response = ""
    with claude_client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=256,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    ) as stream:
        for text_chunk in stream.text_stream:
            full_response += text_chunk
            print(text_chunk, end="", flush=True)
    print()  # Newline after streaming

    # Step 2: Convert complete response to speech
    print("\nGenerating speech...")
    audio_bytes = text_to_speech_streaming(full_response, voice=voice)

    return full_response, audio_bytes

In [ ]:
# Example 3: Streaming voice assistant
user_input = "Give me a motivational message to start my day."
print(f"User: {user_input}\n")

response_text, audio_bytes = voice_assistant_streaming(user_input, voice="English_Insightful_Speaker")
print(f"\nAudio size: {len(audio_bytes):,} bytes")

display(Audio(audio_bytes, rate=32000, autoplay=False))

## Choosing TTS Models

MiniMax offers two TTS models with different tradeoffs:

| Model | Quality | Speed |
|-------|---------|-------|
| `speech-2.8-hd` | High quality (default) | Slower |
| `speech-2.8-turbo` | Good quality | Faster |

Choose `speech-2.8-turbo` for real-time applications where latency matters.

In [ ]:
# Compare TTS model quality
sample_text = "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet."

print("Generating with speech-2.8-hd (high quality)...")
audio_hd = text_to_speech(sample_text, model="speech-2.8-hd")
print(f"HD audio: {len(audio_hd):,} bytes")
display(Audio(audio_hd, rate=32000, autoplay=False))

print("\nGenerating with speech-2.8-turbo (faster)...")
audio_turbo = text_to_speech(sample_text, model="speech-2.8-turbo")
print(f"Turbo audio: {len(audio_turbo):,} bytes")
display(Audio(audio_turbo, rate=32000, autoplay=False))

## Summary

In this notebook, we built a voice assistant using Claude and MiniMax TTS:

- **`text_to_speech()`** — Simple TTS with the non-streaming API
- **`text_to_speech_streaming()`** — Efficient TTS using SSE streaming with hex-decoded audio chunks
- **`voice_assistant()`** — Combined Claude + MiniMax TTS pipeline
- **`voice_assistant_streaming()`** — Progressive text display + TTS synthesis

### Key MiniMax TTS Points

- Audio data is **hex-encoded** (use `bytes.fromhex()`, not `base64`)
- Streaming uses **SSE format**: parse `data:` lines, decode JSON, extract `data.audio`
- Default model: `speech-2.8-hd` | Fast alternative: `speech-2.8-turbo`
- API endpoint: `POST https://api.minimax.io/v1/t2a_v2`

### Next Steps

- **Add STT**: Use Deepgram or Whisper to transcribe voice input for a fully conversational assistant
- **Real-time streaming**: Pipe audio chunks to a local audio player as they arrive for ultra-low latency
- **Custom voices**: Explore MiniMax's voice cloning features for branded voice experiences
- **Multi-turn conversations**: Maintain conversation history with Claude for contextual responses

### Resources

- [MiniMax TTS API Reference](https://platform.minimax.io/docs/api-reference/speech-t2a-http)
- [Anthropic Claude Models](https://docs.anthropic.com/en/docs/about-claude/models/overview)
- [Claude Streaming Guide](https://docs.anthropic.com/en/api/messages-streaming)